### Work with the Survey Manager

In [ ]:
import arcgis
from arcgis.gis import GIS
import datetime
from datetime import date, timedelta
import shutil
import os
from concurrent.futures import ThreadPoolExecutor

gis = GIS(username="survey123_autotest_creator", password="Autotest123")

In [ ]:
survey_manager = arcgis.apps.survey123.SurveyManager(gis)
surveys = survey_manager.surveys
assert len(surveys) > 0

In [ ]:
survey_by_id = survey_manager.get("2ec66822b5364e1fa6e64f4ac7581d2e")
assert len(survey_by_id.properties["title"]) > 0

In [ ]:
forms = gis.content.search("type:form owner:survey123_autotest_creator")
assert len(forms) > 0

In [ ]:
survey_by_item = survey_manager.get(forms[8])
assert len(survey_by_item.properties) > 0

### Work with survey data

In [ ]:
# Download - file formats
dl_formats = ["CSV", "Shapefile", "File Geodatabase"]
for f in dl_formats:
    outfile = survey_by_id.download(f)
    assert outfile != None

In [ ]:
# Download - pandas DataFrame
import pandas as pd

survey_df = survey_by_id.download("DF")
assert len(survey_df) > 0

### Create reports


In [ ]:
# Identify report templates associated with a survey
templates = survey_by_id.report_templates
assert len(templates) > 0

In [ ]:
# Generate a default report template
temp_name = "Sample template " + str(datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
new_template = survey_by_id.create_report_template(template_name=temp_name)
assert new_template != None

In [ ]:
# Check template syntax
check = survey_by_id.check_template_syntax(new_template)
assert check["success"] == True

In [ ]:
# Associate a report template with a survey
upload_template = survey_by_id.upload_report_template(
    template_file=new_template, template_name="PythonAPITemplate"
)
assert upload_template != None
templates = survey_by_id.report_templates
updated_templates = [x.title for x in templates]
assert "PythonAPITemplate" in updated_templates

In [ ]:
# Estimate credits
credits = survey_by_id.estimate(templates[0], where="1=1")
assert credits["success"] == True

In [ ]:
# Create sample report
webmap = gis.content.search(
    query="title:Water Quality Inspection Python API", item_type="Web Map"
)
wm_item = webmap[0]
sample = survey_by_id.create_sample_report(
    templates[0],
    where="objectid=1",
    utc_offset="-07:00",
    report_title="Sample_Report",
    merge_files="none",
    survey_item=survey_by_id,
    webmap_item=wm_item,
    map_scale="10000",
    locale="en",
)
assert sample != None

In [ ]:
# Generate single report
report = survey_by_id.generate_report(templates[0], where="objectid=122")
assert report != None

In [ ]:
# Generate multiple reports
local_batch_reports = survey_by_id.generate_report(
    templates[0],
    where="ws_advisory = 'Yes' and ws_advisory_start_date > '01/01/2020'",
    report_title="SingleReportInstance",
    package_name="ReportPackageNamePython",
)
assert local_batch_reports != None

In [ ]:
# Generate multiple reports and save to your orgainzation
nowstring = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
folder_ID = survey_by_id.properties["ownerFolder"]
org_batch_reports = survey_by_id.generate_report(
    templates[0],
    where="ws_advisory = 'Yes' and ws_advisory_start_date > '01/01/2020'",
    package_name="Test_{0}".format(nowstring),
    folder_id=folder_ID,
)
assert org_batch_reports != None
search_for_batch_reports = gis.content.search("Test_{0}".format(nowstring))
assert len(search_for_batch_reports) > 0

In [ ]:
# Generate report with all possible parameters
folder_ID = survey_by_id.properties["ownerFolder"]
webmap = gis.content.search(
    query="title:Water Quality Inspection Python API", item_type="Web Map"
)
wm_item = webmap[0]
all_params = survey_by_id.generate_report(
    templates[0],
    where="ws_advisory = 'Yes' and ws_advisory_start_date > '01/01/2020'",
    utc_offset="-07:00",
    report_title="All_Params_Report",
    package_name="All_Params_Package",
    output_format="pdf",
    folder_id=folder_ID,
    merge_files="none",
    survey_item=survey_by_id,
    webmap_item=wm_item,
    map_scale="10000",
    locale="en",
)
assert all_params != None

In [ ]:
# Update report template
import tempfile

template_location = os.path.join(
    os.path.abspath(""), "Survey123_resources", "Sample_template.docx"
)
tmpdir = tempfile.TemporaryDirectory()
tmp_folder = tmpdir.name
updated_template = shutil.copy(
    template_location, os.path.join(tmp_folder, "PythonAPITemplate.docx")
)
update = survey_by_id.update_report_template(updated_template)
assert len(update) > 0

In [ ]:
recentReports = survey_by_id.reports
assert len(recentReports) > 0

### Create and publish Surveys

In [ ]:
# Deletes test surveys created


def delete_survey(delete_gis, form_item):
    usr = arcgis.gis.User(delete_gis, delete_gis.users.me.username)
    folder = usr.items(folder=form_item.properties["ownerFolder"])
    [
        x.delete()
        for x in folder
        if x.type != "Feature Service"
        or (x.type == "Feature Service" and "_form" in x.title)
    ]

    [
        x.delete(force=True)
        for x in usr.items(folder=form_item.properties["ownerFolder"])
    ]

    delete_gis.content.folders.get(
        folder=form_item.properties["ownerFolder"], owner=delete_gis.users.me.username
    ).delete()

# Extract test case resources
if not os.path.exists(
    os.path.join(os.path.abspath(""), "Survey123_resources", "publish_dir")
):

    os.mkdir(os.path.join(os.path.abspath(""), "Survey123_resources", "publish_dir"))


shutil.unpack_archive(
    os.path.join(os.path.abspath(""), "Survey123_resources", "publish_tests.zip"),
    os.path.join(os.path.abspath(""), "Survey123_resources", "publish_dir"),
)

#### Create a new survey

In [ ]:
new_survey = survey_manager.create(
    title="Python Test Survey",
    tags=["ArcGIS API for Python, Survey123, Form"],
    summary="This survey was created using the ArcGIS API for Python",
)
assert type(new_survey) == arcgis.apps.survey123.Survey

survey_folder = gis.content.folders._get_or_create(
    folder=new_survey.properties["ownerFolder"], owner=gis.users.me.username
).properties["id"]

usr = arcgis.gis.User(gis, gis.users.me.username)
new_fldr_items = usr.items(folder=survey_folder)
new_fldr_items_types = [x.type for x in new_fldr_items]

assert "Form" in new_fldr_items_types
assert "Feature Service" in new_fldr_items_types

#### Publish to that new survey

In [ ]:
published_survey = new_survey.publish(
    xlsform=os.path.join(
        os.path.abspath(""),
        "Survey123_resources",
        "publish_dir",
        "Hydrant_Inspection_init.xlsx",
    ),
    info={
        "queryInfo": {"mode": "manual", "editEnabled": True, "copyEnabled": True},
        "sentInfo": {"enabled": True, "editEnabled": True, "copyEnabled": True},
        "displayInfo": {
            "map": {
                "coordinateFormat": "usng",
                "home": {"latitude": 34.0568, "longitude": -117.1961, "zoomLevel": 20},
                "preview": {"coordinateFormat": "usng", "zoomLevel": 0},
            }
        },
    },
    create_web_form=True,
    enable_delete_protection=False,
    create_coded_value_domains=True,
    enable_sync=False,
    create_web_map=True,
)


assert type(published_survey) == arcgis.apps.survey123.Survey


fldr_items = usr.items(folder=survey_folder)

fldr_items_types = [x.type for x in fldr_items]


assert "Form" in fldr_items_types

assert "Feature Service" in fldr_items_types

assert "Web Map" in fldr_items_types

for item in fldr_items:

    if (
        item.type == "Feature Service"
        and item
        == [x for x in fldr_items if x.type == "Form"][0].related_items(
            "Survey2Service", "forward"
        )[0]
    ):

        assert "View Service" in item.typeKeywords

#### Update the new survey

In [ ]:
updated_survey = published_survey.publish(
    xlsform=os.path.join(
        os.path.abspath(""),
        "Survey123_resources",
        "publish_dir",
        "Hydrant_Inspection_update.xlsx",
    ),
    schema_changes=True,
)

assert type(updated_survey) == arcgis.apps.survey123.Survey
fldr_items = usr.items(folder=survey_folder)
sub_url = [x for x in fldr_items if x.type == "Form"][0].related_items(
    "Survey2Service", "forward"
)[0]
assert "defects" in [x.properties.name for x in sub_url.tables]

##### TC #109 

Source: https://github.com/ArcGIS/survey123-test-functional/issues/109

Tests to verify that layer ID's are applied as Survey123 expects. 

In [ ]:
tc109_survey = survey_manager.create(title="Test Case 109 Survey")
tc109_survey_published = tc109_survey.publish(
    xlsform=os.path.join(
        os.path.abspath(""), "Survey123_resources", "publish_dir", "Hierachy_123.xlsx"
    ),
    create_web_map=False,
)
tc109_folder = tc109_survey_published.properties["ownerFolder"]

properties = {"Test_Case_109_Survey": 0, "rep3_GP": 1, "rep1_noGP": 2, "rep2_noGP": 3}

for lyr in list(
    tc109_survey_published._ssi.layers + tc109_survey_published._ssi.tables
):
    prop = lyr.properties
    assert prop.name in list(properties.keys())
    assert prop.id == properties[prop.name]

delete_survey(gis, tc109_survey_published)

##### TC #112 
Source: https://github.com/ArcGIS/survey123-test-functional/issues/112

Tests to verify that a survey can be published against a standalone table via submission_url.

In [ ]:
tc112_survey = survey_manager.create(title="Test Case 112 Survey")
tc112_survey_published = tc112_survey.publish(
    xlsform=os.path.join(
        os.path.abspath(""), "Survey123_resources", "publish_dir", "RelatedTable.xlsx"
    ),
    create_web_map=False,
)

assert tc112_survey_published._ssi.id == "e8dc23ed3f654694a7990b5a8641d1f0"

delete_survey(gis, tc112_survey_published)

##### TC #163 
Source: https://github.com/ArcGIS/survey123-test-functional/issues/163

In [ ]:
tc163_survey = survey_manager.create(title="Test Case 163")
tc163_published = tc163_survey.publish(
    xlsform=os.path.join(
        os.path.abspath(""),
        "Survey123_resources",
        "publish_dir",
        "163",
        "TC_163_a.xlsx",
    ),
    create_web_map=False,
)

fields = [x["name"] for x in tc163_published._ssi_layers[0].properties.fields]
assert "text_question" in fields
assert "integer_question" in fields

In [ ]:
tc163_b = tc163_published.publish(
    xlsform=os.path.join(
        os.path.abspath(""),
        "Survey123_resources",
        "publish_dir",
        "163",
        "TC_163_b.xlsx",
    ),
    schema_changes=True,
)
fields = [x["name"] for x in tc163_b._ssi_layers[0].properties.fields]
assert "date_question" in fields

In [ ]:
# Not working need to figure out
# tc163_c = tc163_b.publish(
#     xlsform = os.path.join(os.path.abspath(""), "Survey123_resources", "publish_dir", "163", "TC_163_c.xlsx"),
#     schema_changes = True
# )

# assert tc163_b._ssi_layers[0].properties.hasAttachments is True

In [ ]:
tc163_d = tc163_b.publish(
    xlsform=os.path.join(
        os.path.abspath(""),
        "Survey123_resources",
        "publish_dir",
        "163",
        "TC_163_d.xlsx",
    ),
    schema_changes=True,
)

assert tc163_d._ssi.layers[0].properties.relationships[0].role == "esriRelRoleOrigin"
assert tc163_d._ssi.layers[0].properties.relationships[0].relatedTableId == 1
assert tc163_d._ssi.tables[0].properties.name == "rp1"
assert (
    tc163_d._ssi.tables[0].properties.relationships[0].role == "esriRelRoleDestination"
)
assert tc163_d._ssi.tables[0].properties.relationships[0].relatedTableId == 0

In [ ]:
tc163_e = tc163_d.publish(
    xlsform=os.path.join(
        os.path.abspath(""),
        "Survey123_resources",
        "publish_dir",
        "163",
        "TC_163_e.xlsx",
    ),
    schema_changes=True,
)

fields = [x["name"] for x in tc163_published._ssi.tables[0].properties.fields]
assert "rp1_decimal" in fields

In [ ]:
tc163_f = tc163_d.publish(
    xlsform=os.path.join(
        os.path.abspath(""),
        "Survey123_resources",
        "publish_dir",
        "163",
        "TC_163_f.xlsx",
    ),
    schema_changes=True,
)

fields = [x["name"] for x in tc163_f._ssi_layers[0].properties.fields]
assert "text_question" in fields
assert "integer_question" in fields
assert "date_question" in fields
assert tc163_f._ssi.layers[0].properties.relationships[0].role == "esriRelRoleOrigin"
assert tc163_f._ssi.layers[0].properties.relationships[0].relatedTableId == 1
assert tc163_f._ssi.tables[0].properties.name == "rp1"
assert (
    tc163_f._ssi.tables[0].properties.relationships[0].role == "esriRelRoleDestination"
)
assert tc163_f._ssi.tables[0].properties.relationships[0].relatedTableId == 0
tbl_fields = [x["name"] for x in tc163_f._ssi.tables[0].properties.fields]
assert "rp1_decimal" in tbl_fields
assert "rp1_decimal" in tbl_fields

delete_survey(gis, tc163_f)

##### TC #170 
Source: https://github.com/ArcGIS/survey123-test-functional/issues/170

In [ ]:
# TODO Add repeat count 2 back to the campsite_equipment repeat

tc170_survey = survey_manager.create(title="Test Case 170")

try:
    tc170_published = tc170_survey.publish(
        xlsform=os.path.join(
            os.path.abspath(""),
            "Survey123_resources",
            "publish_dir",
            "170",
            "Nested_Repeat.xlsx",
        ),
        create_web_map=False,
    )
except RuntimeError as r:
    delete_survey(gis, tc170_survey)
    raise AssertionError(str(r))

assert type(tc170_published) == arcgis.apps.survey123.Survey
delete_survey(gis, tc170_published)

##### <a href="https://github.com/ArcGIS/geosaurus/issues/11521">#11521</a>

In [ ]:
tc11521_survey = survey_manager.create(title="Test Case 11521")

try:
    tc11521_published = tc11521_survey.publish(
        xlsform=os.path.join(
            os.path.abspath(""),
            "Survey123_resources",
            "publish_dir",
            "11521",
            "ALLCAPS.xlsx",
        ),
        create_web_map=False,
    )
except RuntimeError as r:
    delete_survey(gis, tc11521_survey)
    raise AssertionError(str(r))

delete_survey(gis, tc11521_survey)

##### TC #172 
Source: https://github.com/ArcGIS/survey123-test-functional/issues/172

In [ ]:
tc172_survey = survey_manager.create(title="Test Case 172 Survey")
tc172_survey_published = tc172_survey.publish(
    xlsform=os.path.join(
        os.path.abspath(""),
        "Survey123_resources",
        "publish_dir",
        "Test_Repeat_pub.xlsx",
    ),
    create_web_map=False,
)

assert tc172_survey_published._ssi.layers[0].properties.name == "Test_Case_172_Survey"
assert tc172_survey_published._ssi.tables[0].properties.name == "example"
assert "test_q" in [
    x["name"] for x in tc172_survey_published._ssi.tables[0].properties.fields
]

delete_survey(gis, tc172_survey_published)

##### TC #232 
Source: https://github.com/ArcGIS/survey123-test-functional/issues/232 ?

In [ ]:
tc232_survey = survey_manager.create(title="Test Case 232 A")
tc232_published = tc232_survey.publish(
    xlsform=os.path.join(
        os.path.abspath(""),
        "Survey123_resources",
        "publish_dir",
        "232",
        "XLSFormTemplate.xlsx",
    ),
    create_web_map=False,
)

survey_adm_url = tc232_published._ssi.url.replace(
    "/rest/services", "/rest/admin/services"
)
survey_flcm = arcgis.features.managers.FeatureLayerCollectionManager(
    url=survey_adm_url, gis=gis, fs=tc232_published._ssi
).properties

assert survey_flcm["capabilities"] == "Create,Editing"
assert survey_flcm["syncEnabled"] is False
assert tc232_published._ssi.layers[0].properties.name == "Test_Case_232_A"
assert [
    x for x in tc232_published._ssi.layers[0].properties.fields if x.name == "yes_no"
][0].domain is not None
assert (
    tc232_published._ssi.layers[0].properties.relationships[0].role
    == "esriRelRoleOrigin"
)
assert tc232_published._ssi.layers[0].properties.relationships[0].keyField == "globalid"
assert tc232_published._ssi.tables[0].properties.name == "repeat1"
assert (
    tc232_published._ssi.tables[0].properties.relationships[0].role
    == "esriRelRoleDestination"
)
assert (
    tc232_published._ssi.tables[0].properties.relationships[0].keyField
    == "parentglobalid"
)

delete_survey(gis, tc232_published)

In [ ]:
# Not implemented

# tc232_survey = survey_manager.create(
#     title="Test Case 232 B"
# )
# tc232_published = tc232_survey.publish(
#     xlsform = os.path.join(os.path.abspath(""), "Survey123_resources", "publish_dir", "232", "XLSFormTemplate.xlsx"),
#     create_web_map = False,
#     info = {
#         "sentInfo": {
#             "enabled": True,
#             "copyEnabled": True
#         }
#     }
# )

# survey_adm_url = tc232_published._ssi.url.replace("/rest/services", "/rest/admin/services")
# survey_flcm = arcgis.features.managers.FeatureLayerCollectionManager(
#     url=survey_adm_url,
#     gis=gis,
#     fs=tc232_published._ssi
#     ).properties

# assert survey_flcm['editorTrackingInfo']['enableEditorTracking'] is True
# assert survey_flcm['editorTrackingInfo']['enableOwnershipAccessControl'] is True
# assert survey_flcm['editorTrackingInfo']['allowOthersToQuery'] is True
# assert survey_flcm['editorTrackingInfo']['allowOthersToUpdate'] is True
# assert survey_flcm['editorTrackingInfo']['allowOthersToDelete'] is True
# assert survey_flcm['editorTrackingInfo']['allowAnonymousToDelete'] is False
# assert survey_flcm['editorTrackingInfo']['allowAnonymousToUpdate'] is False
# assert survey_flcm['syncEnabled'] is False

# delete_survey(tc232_published)

##### TC #245 
Source: https://github.com/ArcGIS/survey123-test-functional/issues/245

In [ ]:
# Needs more investigation

# tc245_survey = survey_manager.create(
#     title="Test Case 245 Survey"
# )
# tc245_published = tc245_survey.publish(
#     xlsform = os.path.join(os.path.abspath(""), "Survey123_resources", "publish_dir", "245", "Domains_and_field_aliases_a.xlsx"),
#     create_web_map = False,

# )

# for domain in [f for f in tc245_published._ssi.layers[0].properties.fields if f.name == "tree_type"][0].domain.codedValues:
#     if domain['code'] == "lodgepole_pine":
#         assert domain['name'] == "Llodgepole Pine"
#     elif domain['code'] == "bristlecone_pine":
#         assert domain['name'] == "Bristlecone Pine"
#     else:
#         assert domain['code'] == "sspruce"

# tc245_updated = tc245_published.publish(
#     xlsform = os.path.join(os.path.abspath(""), "Survey123_resources", "publish_dir", "245", "Domains_and_field_aliases_b.xlsx"),
#     create_web_map = False,
#     schema_changes = True
# )

# ------------------------------------------------ #

# tree_field = [f for f in tc245_updated._ssi.layers[0].properties.fields if f.name == "tree_type"][0]

# assert tree_field['alias'] == "Tree type"

# for domain in tree_field.domain.codedValues:
#     if domain['code'] == "lodgepole_pine":
#         assert domain['name'] == "Lodgepole Pine"
#     elif domain['code'] == "bristlecone_pine":
#         assert domain['name'] == "Bristlecone Pine"
#     else:
#         assert domain['code'] == "spruce"

# delete_survey(gis, tc245_published)

##### TC #248 
Source: https://github.com/ArcGIS/survey123-test-functional/issues/248

In [ ]:
dir_248 = os.path.join(os.path.abspath(""), "Survey123_resources", "publish_dir", "248")


# for form in os.listdir(dir_248):
def tc_248(form):
    if (
        form == "submission_publish_error_3b.xlsx"
        or form == "submission_publish_error_3c.xlsx"
        or form == "submission_publish_error_3d.xlsx"
        or form == "submission_publish_error_3e.xlsx"
    ):
        # Need to figure out
        return
    tc248_survey = survey_manager.create(
        title=f"Test Case 248: {os.path.splitext(form)[0]}"
    )
    try:
        tc248_published = tc248_survey.publish(
            xlsform=os.path.join(dir_248, form), create_web_map=False
        )
        delete_survey(gis, tc248_published)
        raise AssertionError(form)
    except RuntimeError as r:
        if form == "submission_publish_error_3a.xlsx":
            assert (
                dict(eval(str(r).replace("'", '"')))[
                    "submission_publish_error_step1 Errors"
                ]["Field Errors"][0]
                == "Field not found in the feature service for the goose_question question."
            )
        elif form == "submission_publish_error_3f.xlsx":
            assert (
                dict(eval(str(r).replace("'", '"')))[
                    "submission_publish_error_step1 Errors"
                ]["Field Errors"][0]
                == "Field not found in the feature service for the Example1 question."
            )
        delete_survey(gis, tc248_survey)


with ThreadPoolExecutor() as tc:
    tc.map(tc_248, list(os.listdir(dir_248)))

##### TC #269 
Source: https://github.com/ArcGIS/survey123-test-functional/issues/269

In [ ]:
dir_269 = os.path.join(os.path.abspath(""), "Survey123_resources", "publish_dir", "269")


def tc_269(form):

    tc269_survey = survey_manager.create(
        title=f"Test Case 269: {os.path.splitext(form)[0]}"
    )

    try:

        tc269_published = tc269_survey.publish(
            xlsform=os.path.join(dir_269, form), create_web_map=False
        )

        delete_survey(gis, tc269_published)

        raise AssertionError(form)

    except RuntimeError as r:

        assert "Only one geometry field is allowed per layer;" in str(r)

        delete_survey(gis, tc269_survey)


with ThreadPoolExecutor() as tc:

    tc.map(tc_269, os.listdir(dir_269))

##### TC <a href="https://github.com/ArcGIS/geosaurus/issues/12305">#12305</a>

In [ ]:
e_gis = GIS(
    url="https://s123001.geocloud.com/portal",
    username="survey123_autotest_creator",
    password="Autotest123",
)
survey_manager = arcgis.apps.survey123.SurveyManager(e_gis)

tc12305_survey = survey_manager.create(title="Test Case 12305 Survey")
tc12305_survey_published = tc12305_survey.publish(
    xlsform=os.path.join(
        os.path.abspath(""),
        "Survey123_resources",
        "publish_dir",
        "12305",
        "Address.Request.Form.xlsx",
    ),
    create_web_map=False,
)

_form_view_flc = arcgis.features.FeatureLayerCollection(
    tc12305_survey_published._ssi.url, gis=e_gis
)

# Verify spatial reference is correct in _form view successful publishing implicitly verifies parent feature service and _form view spatial reference match
assert _form_view_flc.properties["spatialReference"]["wkid"] == 4326

# Check for providerSDS in parent FS
assert (
    "providerSDS"
    in tc12305_survey_published._si.related_items("Survey2Data", direction="forward")[
        0
    ].typeKeywords
)

# Check for providerSDS in _form view
assert "providerSDS" in tc12305_survey_published._ssi.typeKeywords

delete_survey(e_gis, tc12305_survey)

shutil.rmtree(os.path.join(os.path.abspath(""), "Survey123_resources", "publish_dir"))

##### TC Verify `info` argument completly works

TODO

### Create Survey webhooks

In [ ]:
# Ensure there are no webhooks currently configured

assert len(updated_survey.webhooks) == 0

In [ ]:
# Add a webhook

add_result = updated_survey.add_webhook(
    name="Python Test Webhook",
    payload_url="https://www.arcgis.com",
    trigger_events=["addData", "editData"],
    portal_info=True,
    submitted_record=True,
    user_info=True,
    server_response=True,
    survey_info=True,
    active=True,
)

assert add_result["success"] is True
assert len(updated_survey.webhooks) > 0

added_webhook = [
    x for x in updated_survey.webhooks if x["id"] == add_result["webhookId"]
][0]

assert added_webhook["active"] is True
assert added_webhook["name"] == "Python Test Webhook"
assert added_webhook["url"] == "https://www.arcgis.com"
assert added_webhook["includePortalInfo"] is True
assert added_webhook["includeServiceRequest"] is True
assert added_webhook["includeUserInfo"] is True
assert added_webhook["includeServiceResponse"] is True
assert added_webhook["includeSurveyInfo"] is True
assert added_webhook["events"] == ["addData", "editData"]

In [ ]:
# Update a webhook

update_result = updated_survey.update_webhook(
    webhook_id=add_result["webhookId"],
    name="Python Test Webhook Updated",
    portal_info=False,
    user_info=False,
    survey_info=False,
)

updated_webhook = [
    x for x in updated_survey.webhooks if x["id"] == update_result["webhookId"]
][0]

assert updated_webhook["active"] is True
assert updated_webhook["name"] == "Python Test Webhook Updated"
assert updated_webhook["url"] == "https://www.arcgis.com"
assert updated_webhook["includePortalInfo"] is False
assert updated_webhook["includeServiceRequest"] is True
assert updated_webhook["includeUserInfo"] is False
assert updated_webhook["includeServiceResponse"] is True
assert updated_webhook["includeSurveyInfo"] is False
assert updated_webhook["events"] == ["addData", "editData"]

In [ ]:
# Delete a webhook

delete_result = updated_survey.delete_webhook(update_result["webhookId"])
assert delete_result is True

assert len(updated_survey.webhooks) == 0

In [ ]:
assert upload_template.delete() == True
assert org_batch_reports.delete() == True
assert all_params.delete() == True
delete_survey(gis, updated_survey)